# Лабораторна робота 4 — LASSO-регресія через координатний спуск

**Набір даних:** `kc_house_data.csv`  
**Обмеження:** scikit-learn LASSO **не дозволено** для базових завдань.

## Налаштування

In [ ]:
import sys
!{sys.executable} -m pip install numpy pandas matplotlib --quiet


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

%matplotlib inline


## Теоретичне підґрунтя

Для нормалізованих ознак значення ρᵢ для координатного спуску LASSO:
```
ρᵢ = dot( featureᵢ,  output − (Xw − wᵢ·featureᵢ) )
```
Правило м'якого порогування для wᵢ (i ≥ 1):
```
wᵢ = ρᵢ + λ/2    якщо ρᵢ < −λ/2
wᵢ = 0           якщо −λ/2 ≤ ρᵢ ≤ λ/2
wᵢ = ρᵢ − λ/2    якщо ρᵢ >  λ/2
```
Вільний член (i = 0) **не** регуляризується: w₀ = ρ₀.

## Допоміжні функції — `get_numpy_data` і `predict_output` (повторне використання з Лаб. 3)

Наведено для зручності. Якщо ви виконали Лабораторну роботу 3, можете підставити власну реалізацію.

In [ ]:
def get_numpy_data(dataframe, features, output):
    # 1. Додаємо стовпець одиниць (intercept)
    dataframe = dataframe.copy()
    dataframe['constant'] = 1.0

    # 2. Будуємо матрицю ознак: constant + features
    all_features = ['constant'] + features
    feature_matrix = dataframe[all_features].to_numpy(dtype=float)

    output_array = dataframe[output].to_numpy(dtype=float)
    return feature_matrix, output_array

In [ ]:
def predict_output(feature_matrix, weights):
    """Передбачення через скалярний добуток."""
    return np.dot(feature_matrix, weights)


---
## Завдання 1 — Нормалізація ознак

Реалізуйте `normalize_features(feature_matrix)`, яка ділить кожен стовпець на його норму L2 та повертає `(normalised_features, norms)`.

In [ ]:
def normalize_features(feature_matrix):
    # Обчислюємо норму L2 для кожного стовпця (axis=0)
    norms = np.linalg.norm(feature_matrix, axis=0)
    # Ділимо матрицю на норми
    normalised = feature_matrix / norms
    return normalised, norms

### Перевірка — кожен стовпець `normalised` має дорівнювати [0.6, 0.8]; norms = [5, 10, 15]

In [ ]:
test_matrix = np.array([[3., 6., 9.], [4., 8., 12.]])
normalised, norms = normalize_features(test_matrix)
print('Нормалізована матриця:\n', normalised)
print('Норми:', norms)


Нормалізована матриця:
 [[0.6 0.6 0.6]
 [0.8 0.8 0.8]]
Норми: [ 5. 10. 15.]


---
## Завдання 2 — Один крок координатного спуску

Реалізуйте `lasso_coordinate_descent_step(i, feature_matrix, output, weights, l1_penalty)`. Функція має:
1. Обчислити поточне передбачення.
2. Обчислити ρᵢ.
3. Якщо i = 0: wᵢ = ρᵢ (без штрафу). Якщо i ≥ 1: застосувати правило м'якого порогування.
4. Повернути нове значення ваги.

In [ ]:
def lasso_coordinate_descent_step(i, feature_matrix, output, weights, l1_penalty):
    # 1. Передбачення з поточними вагами
    prediction = predict_output(feature_matrix, weights)
    
    # 2. Обчислення rho_i
    # Формула: rho_i = sum( feature_i * (output - prediction + w_i * feature_i) )
    # Векторно це: dot(feature_i, output - prediction + weights[i] * feature_matrix[:, i])
    rho_i = np.dot(feature_matrix[:, i], (output - prediction + weights[i] * feature_matrix[:, i]))

    # 3. М'яке порогування
    if i == 0:   # Вільний член (intercept) не регуляризується
        new_weight_i = rho_i
    elif rho_i < -l1_penalty / 2:
        new_weight_i = rho_i + l1_penalty / 2
    elif rho_i > l1_penalty / 2:
        new_weight_i = rho_i - l1_penalty / 2
    else:
        new_weight_i = 0.0

    return new_weight_i

### Перевірка — результат має бути ≈ 0.4256

In [ ]:
from math import sqrt
result = lasso_coordinate_descent_step(
    i=1,
    feature_matrix=np.array([[3/sqrt(13), 1/sqrt(10)],
                             [2/sqrt(13), 3/sqrt(10)]]),
    output=np.array([1., 1.]),
    weights=np.array([1., 4.]),
    l1_penalty=0.1
)
print(f'Результат кроку: {result:.4f}  (очікується ≈ 0.4256)')


Результат кроку: 0.4256  (очікується ≈ 0.4256)


---
## Завдання 3 — Циклічний координатний спуск

Реалізуйте `lasso_cyclical_coordinate_descent(feature_matrix, output, initial_weights, l1_penalty, tolerance)`. У кожному циклі оновлюйте ваги 0, 1, …, d−1 по черзі, використовуючи оновлену вагу одразу. Зупиняйтесь, коли максимальна абсолютна зміна ваги за цикл < `tolerance`.

In [ ]:
def lasso_cyclical_coordinate_descent(feature_matrix, output, initial_weights, l1_penalty, tolerance):
    weights = np.array(initial_weights, dtype=float)
    converged = False

    while not converged:
        max_change = 0.0
        for i in range(len(weights)):
            old_weight_i = weights[i]
            # Оновлюємо конкретну вагу за один крок
            weights[i] = lasso_coordinate_descent_step(i, feature_matrix, output, weights, l1_penalty)
            
            # Фіксуємо найбільшу зміну за весь цикл
            max_change = max(max_change, abs(weights[i] - old_weight_i))

        if max_change < tolerance:
            converged = True

    return weights

---
## Завдання 4 — Відбір ознак

Побудуйте нормалізовану матрицю ознак з `kc_house_data.csv`, використовуючи `['sqft_living', 'bedrooms']`. Запустіть координатний спуск з:
```
initial_weights = [0., 0., 0.]
tolerance       = 1.0
```
— `l1_penalty = 1e7`: вкажіть навчені ваги, RSS та які ознаки мають ненульові ваги.  
— `l1_penalty = 1e8`: повторіть. Що змінилося?

In [ ]:
sales = pd.read_csv('../data/kc_house_data.csv')

# Побудуйте та нормалізуйте матрицю ознак
feature_matrix, output = get_numpy_data(sales, ['sqft_living', 'bedrooms'], 'price')
normalised_features, feature_norms = normalize_features(feature_matrix)


In [ ]:
# Запуск з l1_penalty = 1e7
weights_1e7 = lasso_cyclical_coordinate_descent(
    normalised_features, output,
    initial_weights=np.zeros(3),
    l1_penalty=1e7,
    tolerance=1.0
)
print('Ваги (λ=1e7):', weights_1e7)
print('Non-zero features:', ['intercept', 'sqft_living', 'bedrooms'][
    :len([w for w in weights_1e7 if w != 0])  # rough indicator
])

# RSS on the normalised dataset
preds_1e7 = predict_output(normalised_features, weights_1e7)
rss_1e7   = np.sum((preds_1e7 - output) ** 2)
print(f'RSS (λ=1e7): {rss_1e7:.3e}')


Ваги (λ=1e7): [21624997.95951909 63157247.20788956        0.        ]
Non-zero features: ['intercept', 'sqft_living']
RSS (λ=1e7): 1.630e+15


In [ ]:
# Запуск з l1_penalty = 1e8
weights_1e8 = lasso_cyclical_coordinate_descent(
    normalised_features, output,
    initial_weights=np.zeros(3),
    l1_penalty=1e8,
    tolerance=1.0
)
print('Ваги (λ=1e8):', weights_1e8)


Ваги (λ=1e8): [79400304.63764462        0.                0.        ]


**Спостереження:** При збільшенні штрафу $\lambda$ з $10^7$ до $10^8$ алгоритм LASSO повністю занулив вагу ознаки sqft_living, залишивши в моделі лише вільний член (intercept). Це свідчить про те, що сильніша регуляризація витіснила всі предиктори, оскільки їхній внесок у зменшення помилки став меншим за накладений штраф. Таким чином, зі зростанням $\lambda$ модель стає простішою і менш чутливою до конкретних даних, демонструючи ключову властивість LASSO — автоматичний відбір ознак.

---
## ✨ Бонус — Оцінка на ненормалізованих тестових даних

Розбийте дані 80 % / 20 % (`random_state=42`). Побудуйте та нормалізуйте навчальну матрицю ознак з усіма 10 ознаками нижче. Запустіть координатний спуск (`l1_penalty=1e7`, `tolerance=1.0`).

Ознаки: `sqft_living`, `bedrooms`, `bathrooms`, `sqft_lot`, `floors`, `waterfront`, `view`, `condition`, `grade`, `yr_built`.

Щоб оцінювати на ненормалізованих тестових даних, масштабуйте ваги:
```
weights_rescaled = weights / norms
```
Обчисліть RSS на ненормалізованій тестовій матриці ознак. Вкажіть відібрані ознаки.

In [ ]:
FEATURES = [
    'sqft_living', 'bedrooms', 'bathrooms', 'sqft_lot',
    'floors', 'waterfront', 'view', 'condition',
    'grade', 'yr_built'
]

train_data, test_data = train_test_split(sales, test_size=0.2, random_state=42)

# 1. Підготовка даних
train_data, test_data = train_test_split(sales, test_size=0.2, random_state=42)
X_train, y_train = get_numpy_data(train_data, FEATURES, 'price')
X_test, y_test = get_numpy_data(test_data, FEATURES, 'price')

# 2. Нормалізація (важливо: норми беремо з TRAIN і застосовуємо до TEST)
X_train_norm, train_norms = normalize_features(X_train)

# 3. Навчання
initial_weights = np.zeros(len(FEATURES) + 1)
learned_weights = lasso_cyclical_coordinate_descent(
    X_train_norm, y_train, initial_weights, l1_penalty=1e7, tolerance=1.0
)

# 4. Масштабування ваг (щоб працювати з ненормалізованими даними)
# w_original = w_normalized / norm
weights_rescaled = learned_weights / train_norms

# 5. Оцінка на тестових (ненормалізованих) даних
test_predictions = np.dot(X_test, weights_rescaled)
test_rss = np.sum((test_predictions - y_test) ** 2)

# Вивід результатів
print(f"Test RSS: {test_rss:.4e}")
selected_features = [ (['intercept'] + FEATURES)[i] for i, w in enumerate(learned_weights) if w != 0]
print(f"Selected features: {selected_features}")

Test RSS: 3.4249e+14
Selected features: ['intercept', 'sqft_living', 'waterfront', 'view']


**Відібрані ознаки:** Ненульові ваги отримали: intercept (вільний член), sqft_living (житлова площа), waterfront (вид на набережну) та view (краєвид).

**Тестова RSS:** $$3.4249 \times 10^{14}$$

**Інтерпретація:** Вибір цих ознак має чіткий інтуїтивний сенс, оскільки житлова площа традиційно є найсильнішим фактором впливу на ціну нерухомості. Ознаки waterfront та view відображають преміальність розташування будинку, що суттєво підвищує його ринкову вартість порівняно з аналогами. LASSO ефективно відсіяло другорядні параметри (наприклад, кількість спалень чи рік забудови), залишивши лише ті, що створюють найбільшу додану вартість.